# 12.04 · A FastAPI service for long-running agents: runs, resumable streams, cancellation

**Module 12 · Production & Deployment** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 3–4 h |
| 📋 Prerequisites | Lessons 12.01–12.03 and Lesson 08.03 (interrupts and resume). |
| 🔑 Keys | none (optional: `GROQ_API_KEY` for the 💸 LIVE run) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Design** an HTTP API for agents around **background runs**: start (202), follow, cancel, resume, instead of one long request.
2. **Load** settings with `pydantic-settings` so a bad value fails at startup and secrets never print.
3. **Stream** run events over Server-Sent Events with FastAPI's native SSE, and **resume** after a disconnect with `Last-Event-ID`.
4. **Enforce** who can do what: API keys, per-user ownership (404, not 403), and a manager-only approval step.
5. **Make** side effects safe: idempotency keys, one active run per thread (409), back-pressure and budgets (429).
6. **Load test** the service with 20 concurrent users and read the result as SLOs.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [FastAPI: Server-Sent Events](https://fastapi.tiangolo.com/tutorial/server-sent-events/) | `EventSourceResponse`, `ServerSentEvent`, resuming with `Last-Event-ID` (native since 0.135.0) |
| [FastAPI: Settings and environment variables](https://fastapi.tiangolo.com/advanced/settings/) | `pydantic-settings`, `.env` files, settings as a dependency |
| [FastAPI: Lifespan events](https://fastapi.tiangolo.com/advanced/events/) | Building the agent once at startup, cleaning up at shutdown |
| [FastAPI: Security first steps](https://fastapi.tiangolo.com/tutorial/security/first-steps/) | Bearer tokens and the 401 for missing credentials |
| [LangSmith: double texting](https://docs.langchain.com/langsmith/double-texting) | What to do with a second message while a run is active: reject, enqueue, interrupt, rollback |
| [MDN: Using server-sent events](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events) | The wire format and the browser's `EventSource` (auto-reconnect with `Last-Event-ID`) |

> **Version note.** The course env pins `fastapi>=0.141.1,<0.142`: from 0.142 FastAPI requires `opentelemetry-api>=1.44`, while `langgraph-api` (the `langgraph dev` server from Module 02) still pins an older OpenTelemetry, so pip can't install both (checked 2026-10-16). 0.141 already has native SSE. The Docker image in Lesson 12.06 doesn't need `langgraph-api` and uses the latest FastAPI.

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2" "fastapi>=0.141.1,<0.142" "uvicorn>=0.54.0,<1" "pydantic-settings>=2.16.0,<3" "httpx>=0.28.1,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("fastapi", "uvicorn", "pydantic-settings", "httpx",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/12-production-and-deployment/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm12-fastapi-service'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m12-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Part 1 · Why runs, not requests

The obvious API is `POST /chat` → wait → answer. It breaks for agents: runs take seconds to minutes, proxies cut idle connections (often at 60 s), mobile clients drop, users want a stop button, and a refund that waits for a manager's approval can take hours. So the request **starts** a run and returns at once; everything else is about following and steering that run.

| Method and path | Does | Success | Errors |
|---|---|---|---|
| `GET /ok` | Liveness, no auth | 200 | |
| `POST /threads` | New conversation owned by the caller | 201 | 401 |
| `POST /threads/{id}/runs` | Start a run: `{"message"}`, or `{"resume"}` for managers | **202** | 401, 403, 404, 409, 413, 422, 429 |
| `GET /runs/{id}` | Status, answer, timings, usage | 200 | 401, 404 |
| `GET /runs/{id}/stream` | SSE events; `Last-Event-ID` resumes | 200 | 401, 404 |
| `POST /runs/{id}/cancel` | Stop a queued or running run | 202 | 401, 404 |
| `GET /stats` | Run counts and SLOs (managers) | 200 | 401, 403 |

This is the same shape as LangSmith Deployment's Agent Server API (threads, runs, join a stream), built small enough to read: `mastery/service/app.py` is about 200 lines on top of the `RunManager` from Lesson 12.03.

---
## Part 2 · Settings that fail fast

Every setting comes from an environment variable with the `MASTERY_SERVICE_` prefix (or `.env`). `pydantic-settings` converts and validates them when the app starts, so `MASTERY_SERVICE_MAX_CONCURRENT=lots` stops the deploy instead of failing the first request. The API keys are a `SecretStr`: printing the settings shows `**********`.

In [ ]:
# 🧪 Offline: valid settings, a secret that doesn't print, and a bad value caught at startup.
from pydantic import ValidationError

from mastery.service.settings import ServiceSettings

settings = ServiceSettings()
print(settings.mode, settings.max_concurrent, settings.api_keys)
print({k: (p.user_id, p.role) for k, p in settings.principals().items()} and "3 demo principals:",
      sorted({(p.user_id, p.role) for p in settings.principals().values()}))
try:
    ServiceSettings(max_concurrent=0)
except ValidationError as exc:
    print("rejected:", exc.errors()[0]["loc"], exc.errors()[0]["msg"])

**You should now see** `demo 4 **********`, the three demo principals (`C-1001` and `C-1002` as customers, `ops` as manager) and `rejected: ('max_concurrent',) Input should be greater than or equal to 1`.

### What just happened?

The demo keys are fine for a laptop and nothing else: they are in the source code. A real deployment sets `MASTERY_SERVICE_API_KEYS` from a secret store (Lesson 12.06), and Lesson 12.05 replaces static keys with signed tokens.

---
## Part 3 · Start the service inside the notebook

`create_app(build_manager)` builds the FastAPI app; the agent itself is created in the **lifespan** hook, once, at startup. `serve_in_background` runs it with uvicorn in a background thread on a free port, so this notebook can be the client. From a terminal you'd run `uvicorn apps.m12_service.main:app --port 8000` instead (Part 9).

In [ ]:
# 🧪 Offline: the support desk as a service (DEMO mode: offline model, simulated 0.3 s per model call).
import httpx

from mastery.service.app import create_app, serve_in_background
from mastery.service.client import bearer, stream_events
from mastery.service.desk import support_manager

app = create_app(lambda s: support_manager(s, delay_s=0.3), ServiceSettings(), title="Orbita Cloud support desk")
BASE_URL, stop_server = serve_in_background(app)
client = httpx.Client(base_url=BASE_URL, timeout=30)
CUSTOMER, OTHER, MANAGER = "demo-customer", "demo-other", "demo-manager"

print("service at", BASE_URL)
print("GET /ok          ->", client.get("/ok").json())
print("POST /threads    ->", client.post("/threads").status_code, client.post("/threads").json()["detail"])
thread_id = client.post("/threads", headers=bearer(CUSTOMER)).json()["thread_id"]
print("with a key       ->", thread_id)

**You should now see** the service URL (a random free port), `{'ok': True}`, `401 missing or unknown API key`, and a new `thread-…` id.

### What just happened?

`/ok` needs no key, so a load balancer or Docker health check can call it. Everything else goes through one dependency (`caller`) that maps the bearer key to a user and a role; FastAPI's `HTTPBearer` returns the credentials and our code answers 401 with a `WWW-Authenticate` header when they are missing or unknown.

---
## Part 4 · A run, end to end, over SSE

Start a run, then follow it. Each SSE event has a type (`event:`), a JSON payload (`data:`) and a position (`id:`). The stream sends a `metadata` event, `token`s as the answer is written, a `step` per graph node, and ends with `end` (or `interrupt` or `error`).

In [ ]:
# 🧪 Offline: ask for invoices and print the stream as it arrives.
started = client.post(f"/threads/{thread_id}/runs", json={"message": "I'm customer C-1002. Please show my invoices."},
                      headers=bearer(CUSTOMER))
print(started.status_code, started.json())
run_id = started.json()["run_id"]

events = stream_events(client, run_id, CUSTOMER)
print("events:", [f"{e.id}:{e.event}" for e in events][:6], "…", f"{events[-1].id}:{events[-1].event}")
print("answer:", "".join(e.data["text"] for e in events if e.event == "token")[:120])
status = client.get(f"/runs/{run_id}", headers=bearer(CUSTOMER)).json()
print({k: status[k] for k in ("status", "queue_s", "ttft_s", "duration_s", "usage", "cost_usd")})

**You should now see** `202 {'run_id': 'run-…', …, 'status': 'running'}`, events from `0:metadata` to `11:end`, `Invoices for C-1002 (Team plan): INV-2001 …`, and a TTFT and duration of about 0.9 s (three simulated model calls of 0.3 s), with about 1,250 tokens and a cost estimate of $0.0001.

### What just happened?

The POST returned before the agent did anything; the stream delivered the work as it happened. `GET /runs/{id}` gives the same result to clients that can't stream (or come back later), and the timings are exactly the SLI inputs of Lesson 12.01.

---
## Part 5 · Disconnect, then resume with `Last-Event-ID`

Connections drop. Because the run lives on the server and its events are numbered, a client that reconnects with the **last id it saw** gets only what it missed. Browsers' `EventSource` does this automatically; here we do it by hand: read three events, hang up, reconnect.

In [ ]:
# 🧪 Offline: hang up after 3 events, reconnect from the last id we saw.
run_id = client.post(f"/threads/{thread_id}/runs", json={"message": "And my plan?"}, headers=bearer(CUSTOMER)).json()["run_id"]
first = stream_events(client, run_id, CUSTOMER, limit=3)
print("before the drop:", [e.id for e in first])
rest = stream_events(client, run_id, CUSTOMER, last_event_id=first[-1].id)
print("after reconnect:", [e.id for e in rest][:5], "…", rest[-1].id, rest[-1].event)
ids = [int(e.id) for e in first + rest]
print("no gaps, no duplicates:", ids == list(range(len(ids))))

**You should now see** ids `['0', '1', '2']`, then `3` onwards to the `end` event, and `no gaps, no duplicates: True`.

### What just happened?

The server ignored the disconnect (the run kept going) and replayed from the event log. Two production details: the log must outlive the process for this to survive a restart (Lesson 12.05 stores runs in Postgres), and proxies must not buffer the stream (FastAPI sets `X-Accel-Buffering: no` and sends keep-alives; our endpoint adds a comment every 10 s while a run is quiet).

---
## Part 6 · Ownership and the approval step

Two rules protect money. A customer only sees **their** threads and runs: anything else is **404**, not 403, so nobody can probe which ids exist. And a refund above $50 pauses (`interrupt`); only a **manager** key can resume it.

In [ ]:
# 🧪 Offline: a $199 refund pauses; the other customer can't see it; the customer can't approve it; the manager can.
refund_thread = client.post("/threads", headers=bearer(CUSTOMER)).json()["thread_id"]
REFUND = "I'm customer C-1002. I want a refund of $199 for INV-2001, we moved to another vendor."
run_id = client.post(f"/threads/{refund_thread}/runs", json={"message": REFUND}, headers=bearer(CUSTOMER)).json()["run_id"]
paused = stream_events(client, run_id, CUSTOMER)[-1]
print("last event:", paused.event, "→", paused.data["interrupts"][0]["question"])

print("other customer reads it  ->", client.get(f"/runs/{run_id}", headers=bearer(OTHER)).status_code)
print("customer approves itself ->", client.post(f"/threads/{refund_thread}/runs", json={"resume": "approve"},
                                                headers=bearer(CUSTOMER)).status_code)
resumed = client.post(f"/threads/{refund_thread}/runs", json={"resume": "approve"}, headers=bearer(MANAGER)).json()
print("manager approves         ->", stream_events(client, resumed["run_id"], MANAGER)[-1].data["answer"])
print("refunds issued:", [(r["invoice_id"], r["amount"]) for r in app.state.manager.desk.refunds])

**You should now see** `interrupt → Approve a $199.00 refund (over $50)? …`, `404` for the other customer, `403` for the self-approval, the manager's run ending `Refunded $199.00 for INV-2001 (approved by billing manager)…`, and one refund in the list.

### What just happened?

The approval is just another run on the same thread with `{"resume": ...}`; the graph's checkpointer remembered exactly where it paused (Lesson 08.03). Authorization lives in the API (who may resume), and identity lives in the agent's context: each run acts as the thread **owner** (`SupportContext`), so even the manager's resume can only refund C-1002's invoice.

---
## Part 7 · Double submits and double texting

Networks retry. A user taps "send" twice. Without protection, the refund request runs twice. Two mechanisms:

- **Idempotency key**: the client sends `Idempotency-Key: <uuid>` with each logical request; the same key from the same user returns the **original** run.
- **One active run per thread**: a second message while a run is active gets **409**. LangSmith Deployment calls this *double texting* and offers four strategies: reject, enqueue (its default), interrupt, rollback. We implement reject; enqueue is an exercise.

In [ ]:
# 🧪 Offline: the same request twice with one idempotency key, then a second message mid-run.
t = client.post("/threads", headers=bearer(OTHER)).json()["thread_id"]           # C-1001 this time
headers = bearer(OTHER) | {"Idempotency-Key": "7f1c2b9e-refund-INV-1003"}
body = {"message": "C-1001: I was charged twice on Oct 1. Please refund the duplicate invoice INV-1003 ($49)."}
a = client.post(f"/threads/{t}/runs", json=body, headers=headers).json()
b = client.post(f"/threads/{t}/runs", json=body, headers=headers).json()
print("same run for the retry:", a["run_id"] == b["run_id"])
busy = client.post(f"/threads/{t}/runs", json={"message": "hello?"}, headers=bearer(OTHER))
print("second message mid-run ->", busy.status_code, busy.json()["detail"][:60])
print("answer:", stream_events(client, a["run_id"], OTHER)[-1].data["answer"])
print("refunds for INV-1003:", sum(r["invoice_id"] == "INV-1003" for r in app.state.manager.desk.refunds))

**You should now see** `same run for the retry: True`, `409 thread … already has an active run`, the answer `Refunded $49.00 for INV-1003 …` (under $50: no approval needed) and `refunds for INV-1003: 1`.

### What just happened?

The idempotency key turned a dangerous retry into a harmless one. Keys must be generated by the **client** per logical action (a UUID when the user presses the button), and stored by the server long enough to cover retries; ours live in memory, Lesson 12.05 moves them to the database.

---
## Part 8 · Cancel a running run

`POST /runs/{id}/cancel` cancels the background task (Lesson 12.03, Part 5); the stream ends with `end` and `status: cancelled`.

In [ ]:
# 🧪 Offline: start, cancel after the first event, read how it ended.
t = client.post("/threads", headers=bearer(CUSTOMER)).json()["thread_id"]
run_id = client.post(f"/threads/{t}/runs", json={"message": "I'm customer C-1002. Please show my invoices."},
                     headers=bearer(CUSTOMER)).json()["run_id"]
print("cancel ->", client.post(f"/runs/{run_id}/cancel", headers=bearer(CUSTOMER)).json())
print("stream ->", [(e.event, e.data.get("status")) for e in stream_events(client, run_id, CUSTOMER)][-1])

**You should now see** `{'run_id': 'run-…', 'status': 'cancelled'}` and `('end', 'cancelled')`.

---
## Part 9 · 20 concurrent users

`mastery.service.loadtest.run_load` plays 20 users at once over real HTTP and SSE: each creates a thread, starts a run, follows the stream, and records TTFT and duration as `RunRecord`s, so the report reads as SLOs. Two service configurations, same simulated latency:

In [ ]:
# 🧪 Offline: the same 20-user burst against max_concurrent=2 and max_concurrent=20 (simulated latency, DEMO keys).
from mastery.service.loadtest import run_load

rows = []
for limit in (2, 20):
    test_app = create_app(lambda s: support_manager(s, delay_s=0.3), ServiceSettings(max_concurrent=limit))
    url, stop_test = serve_in_background(test_app)
    report = await run_load(url, users=20, key=CUSTOMER)
    stop_test()
    rows.append({"max_concurrent": limit, "wall_s": report["wall_s"], **report["by_status"],
                 "ttft_p50_s": report["ttft_p50"], "ttft_p95_s": report["ttft_p95"],
                 "ttft_slo_met": report["slos"][1]["met"]})
table(rows, ["max_concurrent", "wall_s", "ok", "error", "rejected", "ttft_p50_s", "ttft_p95_s", "ttft_slo_met"])

**You should now see** 20 `ok` runs in both rows: with `max_concurrent=2` about 8 s of wall time and a TTFT p95 around 7 s (`ttft_slo_met` False); with 20 about 1 s and a TTFT p95 around 1.1 s (True).

### What just happened?

The measurement is end to end: HTTP, auth, queueing, the agent, SSE parsing. That is what users feel and what Project 12's load report records. In LIVE mode the limit that matters is the provider's quota (Lesson 12.03, Part 6), so keep load tests in DEMO mode and run a handful of LIVE runs for real latencies.

---
## Part 10 · Running it for real

The same app ships as `apps/m12_service/main.py`. From the repository root, with the course env active:

```bash
uvicorn apps.m12_service.main:app --port 8000
```

Then, in a second terminal (Git Bash or PowerShell; on Windows use `curl.exe`, not the PowerShell alias):

```bash
curl.exe -s -X POST -H "Authorization: Bearer demo-customer" http://127.0.0.1:8000/threads
curl.exe -s -X POST -H "Authorization: Bearer demo-customer" -H "Content-Type: application/json" -d "{\"message\": \"I'm C-1002, show my invoices\"}" http://127.0.0.1:8000/threads/THREAD_ID/runs
curl.exe -N -H "Authorization: Bearer demo-customer" http://127.0.0.1:8000/runs/RUN_ID/stream
python -m mastery.service.loadtest --users 20
```

Open `http://127.0.0.1:8000/docs` for the interactive OpenAPI page FastAPI generates. For LIVE mode set `MASTERY_SERVICE_MODE=live` in `.env` (with `GROQ_API_KEY`) and restart.

In [ ]:
# 🧪 Offline: stop the notebook's server (the next lessons start their own).
client.close()
stop_server()
print("stopped")

### 💸 Optional: one LIVE run through the service

> 💸 Calls `gpt-oss-20b` through the desk (3–4 model calls with tool schemas: about 4–6K tokens of your free-tier quota). Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
if HAVE_GROQ_KEY:
    live_app = create_app(support_manager, ServiceSettings(mode="live"))
    live_url, stop_live = serve_in_background(live_app)
    with httpx.Client(base_url=live_url, timeout=120) as live_client:
        t = live_client.post("/threads", headers=bearer(CUSTOMER)).json()["thread_id"]
        run_id = live_client.post(f"/threads/{t}/runs", json={"message": "I'm customer C-1002. Please show my invoices."},
                                  headers=bearer(CUSTOMER)).json()["run_id"]
        events = stream_events(live_client, run_id, CUSTOMER)
        status = live_client.get(f"/runs/{run_id}", headers=bearer(CUSTOMER)).json()
    stop_live()
    print(events[-1].event, "·", (status["answer"] or status["error"] or "")[:160])
    print({k: status[k] for k in ("ttft_s", "duration_s", "usage", "cost_usd")})
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** an `end` event with C-1002's invoices in the model's words, a TTFT of one to a few seconds and the run's real token usage and cost. If `usage` is empty, the provider didn't report usage on the stream: the run still worked, but the ledger can't charge it (note it in your research log).

### What just happened?

Same API, same client, real model: everything you tested offline carries over unchanged, which is the point of building DEMO and LIVE behind one interface.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Every request is 422 with a "query" field for the key | `from __future__ import annotations` in the app module with local `Annotated` aliases | Remove it there (FastAPI resolves annotations at runtime) |
| The stream arrives all at once | A proxy or the client buffers | `X-Accel-Buffering: no` (FastAPI sets it), `curl -N`, iterate lines |
| Client reconnects and gets everything again | `Last-Event-ID` not sent, or ids not stable | Send the last id; number events per run |
| Two refunds for one click | No idempotency key, or a key generated per retry | Client generates one key per logical action |
| 409 on a normal second message | The previous run is still active (or paused) | Wait for `end`, cancel it, or resume the interrupt first |
| `pip` refuses fastapi 0.142+ | `opentelemetry-api` conflict with `langgraph-api` | Keep `<0.142` in the course env; latest in the Docker image |
| Port already in use | A previous server is still running | `stop_server()`, or let `port=0` choose |

## ⚠️ Production notes

> ⚠️ **Production note: in-memory state ends with the process.**
> Threads, runs, events and idempotency keys live in one Python process here: a restart loses them and two workers don't share them. Lesson 12.05 moves them to Postgres; Lesson 12.06 runs it in Compose.

> ⚠️ **Production note: put TLS, request limits and CORS in front.** (Part 10)
> Don't expose uvicorn directly to the internet. Terminate TLS at a reverse proxy or the platform, limit request body size there too, and allow only your UI's origin with CORS. Static API keys are for service-to-service calls; users get short-lived tokens (Lesson 12.05).

## 🏋️ Exercises

1. **(Easy)** Add `GET /threads/{id}/runs` that lists a thread's runs (owner or manager only). Return 404 for others.
2. **(Medium)** Implement the **enqueue** double-texting strategy behind a setting: instead of 409, the second message starts after the active run ends.
3. **(Medium)** Send `retry: 3000` on the first SSE event so browsers wait 3 s before reconnecting. Which `ServerSentEvent` field is it?
4. **(Stretch)** Write a 15-line HTML page with `EventSource` that shows tokens as they arrive. Why can't `EventSource` send the `Authorization` header, and what do people do instead?

<details>
<summary><b>Show solutions</b></summary>

**1.** Inside `create_app`:
```python
@app.get("/threads/{thread_id}/runs")
async def list_runs(thread_id: str, who: Caller, runs: Manager):
    owned_thread(thread_id, who)
    return [r.public() for r in runs.runs.values() if r.thread_id == thread_id]
```

**2.** In `RunManager.submit`, when the thread is busy and the strategy is `enqueue`, create the run as usual but make its task first `await self.wait(active)`; the semaphore and queue limits still apply. Keep 409 as the default for side-effecting agents: a queued "refund" that runs after the user changed their mind is worse than an error.

**3.** `ServerSentEvent(retry=3000, ...)`: the `retry` field, in milliseconds.

**4.**
```html
<pre id="out"></pre>
<script>
  const es = new EventSource("/runs/RUN_ID/stream?token=SHORT_LIVED_TOKEN");
  es.addEventListener("token", e => out.textContent += JSON.parse(e.data).text);
  es.addEventListener("end", () => es.close());
</script>
```
`EventSource` has no option for custom headers. Common answers: a cookie session on the same origin, a short-lived token in the URL (scoped to one run, expiring in a minute), or `fetch()` with a streaming body reader instead of `EventSource`.
</details>

## 🔎 Docs drill (5 min)

Which LangSmith Deployment double-texting strategy is the default, and what does `rollback` do with the first run?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Double texting](https://docs.langchain.com/langsmith/double-texting).
2. The page lists four strategies and says **enqueue** is the default.
3. `rollback` halts the current run and **reverts all its progress, including its input**, then processes the new input as a fresh run (`interrupt` halts it but keeps what it did).

</details>

## ✅ Checkpoint

- [ ] I can explain why agents need background runs and which status codes each endpoint returns.
- [ ] Settings load from the environment, fail fast, and never print secrets.
- [ ] I followed a run over SSE and resumed after a disconnect with `Last-Event-ID`, with no gaps or duplicates.
- [ ] Other users get 404; only a manager can approve a paused refund.
- [ ] An idempotency key prevented a double refund; a second message mid-run got 409.
- [ ] I cancelled a run and load-tested 20 concurrent users.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Re-read [FastAPI: Server-Sent Events](https://fastapi.tiangolo.com/tutorial/server-sent-events/) and the [release notes](https://fastapi.tiangolo.com/release-notes/), and log in [`docs/research-log.md`](../../docs/research-log.md) any change to SSE (keep-alive interval, headers) and whether the OpenTelemetry pin conflict is resolved.

## Summary

- Agents are served as background runs: POST starts (202), SSE follows, cancel stops, resume continues.
- Settings via `pydantic-settings`: validated at startup, secrets as `SecretStr`.
- Numbered events + `Last-Event-ID` make streams resumable; the run lives on the server.
- 404 for other people's resources, a manager role for approvals, the owner's identity inside the agent.
- Idempotency keys and one active run per thread make retries and double submits safe.
- Load-test end to end and read the result as SLOs.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 12-05-persistence-and-auth.ipynb** → [`12-05-persistence-and-auth.ipynb`](12-05-persistence-and-auth.ipynb).

[Module 12 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)